# Week 1 — Checkpoint 6: Written Conclusion
### Dataset: `house_sale.csv` (bina.az — real estate sale listings)

**Goal:** Based on the cleaning, outlier handling, feature engineering, and visualization work done in Checkpoints 1–5, present 5 key insights supported by data.

## 1. Restoring the previous steps

In [1]:
import pandas as pd
import numpy as np
import re

pd.set_option('display.max_columns', 70)

df = pd.read_csv('house_sale.csv')

df_clean = df.copy()
df_clean = df_clean.drop(columns=["Binanın növü", "hour_y"])
df_clean["featured_flag"] = df_clean["featured"].notnull()
df_clean["vip_flag"] = df_clean["vip"].notnull()
df_clean["mortgage_flag"] = df_clean["mortgage"].notnull()
df_clean["bill_of_sale_flag"] = df_clean["bill_of_sale"].notnull() | df_clean["Çıxarış"].notnull()
df_clean = df_clean.drop(columns=["featured", "vip", "mortgage", "İpoteka", "bill_of_sale", "Çıxarış", "repair"])
df_clean["Təmir"] = df_clean["Təmir"].fillna("Unknown")

def seller_type(row):
    if pd.notnull(row["shop_name"]):
        return "Agency"
    elif pd.notnull(row["owner_name"]):
        return "Individual"
    return "Unknown"

df_clean["seller_type"] = df_clean.apply(seller_type, axis=1)
for col in ["shop_name", "shop_title", "owner_name", "owner_title"]:
    df_clean[col] = df_clean[col].fillna("N/A")
df_clean["products_label"] = df_clean["products_label"].fillna("Regular listing")
df_clean["description"] = df_clean["description"].fillna("")

def parse_area(val):
    if pd.isnull(val):
        return np.nan
    val = val.strip()
    if "sot" in val:
        return float(val.replace("sot", "").strip().replace(",", ".")) * 100
    return float(val.replace("m²", "").strip().replace(" ", ""))

df_clean["Sahə_numeric"] = df_clean["Sahə"].apply(parse_area)
df_clean["unit_price_calculated"] = (df_clean["price"] / df_clean["Sahə_numeric"]).round(2)
df_clean = df_clean.drop(columns=["unit_price"])

applicable_categories = ["Köhnə tikili", "Yeni tikili", "Ofis", "Həyət evi/Bağ evi"]
mask = df_clean["Kateqoriya"].isin(applicable_categories)
medians = df_clean.loc[mask].groupby("Kateqoriya")["Otaq sayı"].median()
for cat, med in medians.items():
    idx = df_clean[(df_clean["Kateqoriya"] == cat) & (df_clean["Otaq sayı"].isnull())].index
    df_clean.loc[idx, "Otaq sayı"] = med

df_clean = df_clean[df_clean["price"] >= 1000].copy()

def iqr_bounds(series, k=1.5):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr

def cap_grouped(df, col, group_col="Kateqoriya", k=1.5):
    capped = df[col].copy()
    flag = pd.Series(False, index=df.index)
    for cat, group in df.groupby(group_col):
        low, high = iqr_bounds(group[col].dropna(), k)
        idx = group.index
        below = df.loc[idx, col] < low
        above = df.loc[idx, col] > high
        flag.loc[idx[below | above]] = True
        capped.loc[idx[below]] = low
        capped.loc[idx[above]] = high
    return capped, flag

df_clean["price_capped"], df_clean["is_outlier_price"] = cap_grouped(df_clean, "price")
df_clean["Sahə_numeric_capped"], _ = cap_grouped(df_clean, "Sahə_numeric")
df_clean["unit_price_capped"], _ = cap_grouped(df_clean, "unit_price_calculated")

floor_split = df_clean["Mərtəbə"].str.split("/", expand=True)
df_clean["floor_current"] = pd.to_numeric(floor_split[0].str.strip(), errors="coerce")
df_clean["floor_total"] = pd.to_numeric(floor_split[1].str.strip(), errors="coerce")
df_clean["is_ground_floor"] = df_clean["floor_current"] == 1

def loc_type(loc):
    if pd.isnull(loc):
        return "Unknown"
    match = re.search(r"\s(r|m|q)\.$", loc.strip())
    mapping = {"r": "District center", "m": "Micro-district", "q": "Settlement"}
    return mapping.get(match.group(1), "Other") if match else "Other"

df_clean["location_type"] = df_clean["location"].apply(loc_type)

print("df_clean is ready:", df_clean.shape)


df_clean is ready: (100765, 56)


## 2. Key Insights

Below, for each insight I first show the concrete number that supports it, then discuss what it means.

### Insight 1 — The market is almost entirely the apartment segment

Together, `Yeni tikili` (new construction) and `Köhnə tikili` (older buildings) account for **~75.5%** of all listings, while the remaining categories (House/Garden, Land, Object, Office, Garage) make up only ~24.5%.

In [2]:
category_share = (df_clean["Kateqoriya"].value_counts(normalize=True) * 100).round(1)
category_share


,proportion
Kateqoriya,
Yeni tikili,57.5
Köhnə tikili,18.0
Həyət evi/Bağ evi,15.5
Torpaq,4.9
Obyekt,4.0
Ofis,0.2
Qaraj,0.1


 the bina.az sale market in Baku is, in practice, an apartment market — any analysis or future pricing model should focus primarily on these two categories (`Yeni tikili`, `Köhnə tikili`), while the other categories (Garage, Office) are niche segments that may need separate modeling (different price dynamics, see Insight 2).

### Insight 2 — Property category alone can shift price by up to 20x

Looking at median price by category, the most expensive (`Obyekt`, commercial) is **20 times** the least expensive (`Qaraj`, garage).

In [3]:
category_median_price = df_clean.groupby("Kateqoriya")["price_capped"].median().sort_values(ascending=False)
category_median_price


,price_capped
Kateqoriya,
Obyekt,590000.0
Ofis,475000.0
Yeni tikili,245000.0
Həyət evi/Bağ evi,235000.0
Torpaq,150000.0
Köhnə tikili,145000.0
Qaraj,30000.0


In [4]:
ratio = category_median_price.max() / category_median_price.min()
print(f"Most expensive / cheapest category ratio: {ratio:.1f}x")


Most expensive / cheapest category ratio: 19.7x


**Takeaway:** this numerically confirms why I applied IQR **grouped by `Kateqoriya`** in Checkpoint 3 — an `Obyekt` price that looks like an "outlier" on a global scale is completely normal within its own category. Category is one of the single strongest determinants of price.

### Insight 3 — Location type (district center / micro-district / settlement) has a substantial effect on price

32.4% of listings are in `Settlement`-type locations, yet their median price is **~45% lower** than in `District center` locations.

In [5]:
location_median_price = df_clean.groupby("location_type")["price_capped"].median().sort_values(ascending=False)
location_share = (df_clean["location_type"].value_counts(normalize=True) * 100).round(1)

print("Median price by location type:")
print(location_median_price)
print()
print("Share by location type (%):")
print(location_share)

Median price by location type:
location_type
District center    250000.0
Micro-district     230000.0
Settlement         173000.0
Other               87000.0
Name: price_capped, dtype: float64

Share by location type (%):
location_type
Micro-district     53.7
Settlement         32.4
District center    13.5
Other               0.3
Name: proportion, dtype: float64


In [6]:
diff_pct = (location_median_price["District center"] - location_median_price["Settlement"]) / location_median_price["Settlement"] * 100
print(f"'District center' price is {diff_pct:.0f}% higher than 'Settlement'")


'District center' price is 45% higher than 'Settlement'


**Takeaway:** the `city` column alone ("bakı" for 99.6% of rows) couldn't explain this price difference — but the `location_type` feature I created in Checkpoint 4 shows that even within Baku, proximity to the center substantially affects price. This makes it a much more useful variable than `city` for price-estimation models.

### Insight 4 — Room count is the strongest simple (non-derived) price indicator

Among the numeric columns (per the Checkpoint 5 correlation heatmap), `Otaq sayı` (number of rooms) has a **0.57** correlation with `price_capped` — stronger than other "raw" variables (not derived from price itself) such as `Sahə` (0.26) or `floor_ratio` (-0.05).

In [7]:
corr_room_price = df_clean[["Otaq sayı", "price_capped"]].corr().iloc[0, 1]
corr_area_price = df_clean[["Sahə_numeric_capped", "price_capped"]].corr().iloc[0, 1]
print(f"Room count vs price correlation: {corr_room_price:.2f}")
print(f"Area vs price correlation: {corr_area_price:.2f}")


Room count vs price correlation: 0.57
Area vs price correlation: 0.26


**Takeaway:** interestingly, `Sahə` (physical size) correlates with price more weakly than `Otaq sayı` (a human-perceived measure of size). This suggests `Otaq sayı` could be a more reliable simple indicator than `Sahə` for an initial price model — though ideally both would be used together.

### Insight 5 — Agency-listed properties are priced ~40% higher, on average, than individually-listed ones

71.5% of listings are posted by an agency (`shop_name` filled), and 27.9% by an individual seller. Agency listings have systematically higher price and price-per-m² than individual listings.

In [8]:
seller_stats = df_clean.groupby("seller_type").agg(
    median_price=("price_capped", "median"),
    median_unit_price=("unit_price_capped", "median"),
    share_pct=("seller_type", "size")
)
seller_stats["share_pct"] = (seller_stats["share_pct"] / len(df_clean) * 100).round(1)
seller_stats


,median_price,median_unit_price,share_pct
seller_type,,,
Agency,235000.0,2286.82,71.5
Individual,167000.0,1934.96,27.9
Unknown,395352.0,3100.00,0.7


In [9]:
agency_premium = (seller_stats.loc["Agency", "median_price"] - seller_stats.loc["Individual", "median_price"]) / seller_stats.loc["Individual", "median_price"] * 100
print(f"Agency listings' median price is {agency_premium:.0f}% higher than individual listings")

Agency listings' median price is 41% higher than individual listings


**Takeaway:** there are a few possible explanations — (a) agencies tend to handle more expensive/larger properties, (b) agencies may list slightly above market value and negotiate down, or (c) both. It's important to note this gap reflects a **correlation**, not necessarily **causation** — there's a relationship between `seller_type` and `price`, but it doesn't mean "going through an agency makes you pay more," since the underlying property itself may simply differ.

## Checkpoint 6 — Final Summary

1. **Market structure:** ~75.5% of the Baku bina.az sale market is in the apartment segment (new/old construction).
2. **Category = price scale:** property category alone can shift price by up to ~20x (Object vs Garage) — meaning any price analysis must be segmented by category.
3. **Location matters, even within Baku:** `location_type` (district center/micro-district/settlement) creates up to ~45% of price variation — a richer signal than the `city` column.
4. **Room count is the strongest simple indicator:** among raw variables, `Otaq sayı` (0.57) has the strongest relationship with price, even outperforming `Sahə` (0.26).
5. **Seller type correlates with price, but causation should be interpreted carefully:** agency listings are ~40% pricier than individual ones — worth accounting for `seller_type` in future analysis, but not treating it in isolation as a "price-increasing factor."

**Additional note (on data quality):** these results are only reliable because of the careful cleaning done in Checkpoints 2–3 — the raw dataset contained blatant errors like a 600,000,000 AZN listing, which, if left uncorrected, would have seriously distorted every median and correlation figure above.